# SDF for quadratic BSpline


In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
from typing import NamedTuple
import numpy as np
from numpy.random import random as nprand
from numpy.typing import NDArray
import ipywidgets as wg
import k3d

from utils import npvec, arr, arrgs, garr, unflat, f32, disquance, normalize

In [ ]:
def pladd(plot, *args):
    for a in args:
        plot.__iadd__(a)

# Bezier

A segment of $B^{(2)}$-Spline between $c_{j-1}, c_{j}, c_{j+1}$ is equivalent to $B^{(2)}$-ezier with $c_1 = ½(c_{j-1} + c_j), c_2 = c_j, c_3 = ⅓(c_{j+1} + c_j)$

Centered around $c2$ and $t \in [-0.5, +0.5]$

- $c_2 \to 0$
- $c_1 \to v_1 = c_1 - c_2$
- $c_3 \to v_3 = c_3 - c_2$
- $v_a = v_3 + v_1$ — parabolic axis
- $v_d = v_3 - v_1$ — parabolic direction
- $c_a = S(0) - c_2 = .25 v_a$ ­— apex of the parabola (closest to $c_2$, max curvature)
- $p_s = p - c_2$
- $p_c = c_a - p_s$ ­— sample relative to apex

$$
S(t') - c_2 =
\big[1, t', t'^2 \big]
\begin{bmatrix}
c_a \\
v_d \\
v_a \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t') =
\big[1, t' \big]
\begin{bmatrix}
v_d \\
2 v_a \\
\end{bmatrix}
$$

### Projecting

$$
[1, t, t^2, t^3]
\begin{pmatrix}
p_c·v_d \\
2 p_c·v_a + v_d·v_d\\
3 v_a·v_d \\
2 v_a·v_a \\
\end{pmatrix}
=0
\tag{perpendicularity}
$$

### Orientation

- $O$ — vector from projection to sample
- $T$ — tangential vector at projection point
- $[O×T]_z = O_x T_y - O_y T_x$

$$
\begin{cases}
[O×T]_z > 0 \implies \text{O points to right-hand side} \\
[O×T]_z < 0 \implies \text{O points to left-hand side} \\
\tag{orientation}
\end{cases}
$$

### Horizon

$$
\begin{cases}
p_s · v_1 > v_1 · v_1 \implies t'_{prj} < -0.5
\\
p_s · v_3 > v_3 · v_3 \implies t'_{prj} > +0.5
\end{cases}
\tag{visibility}
$$


In [ ]:
class Bezielt:
    """Centered at c_2 and t=-0.5..+0.5"""

    c2: npvec
    ca: npvec
    v1: npvec
    v3: npvec
    va: npvec
    vd: npvec

    def __init__(self, bcontrols: NDArray):
        p1 = bcontrols[0]
        p2 = bcontrols[1]
        p3 = bcontrols[2]
        self.c2 = p2
        self.v1 = (p1 - p2) * 0.5
        self.v3 = (p3 - p2) * 0.5
        self.va = self.v3 + self.v1
        self.vd = self.v3 - self.v1
        self.ca = 0.25 * self.va

    def loc(self, point: npvec) -> npvec:
        return point - self.c2

    def glb(self, point: npvec) -> npvec:
        return point + self.c2

    def point(self, t: float) -> npvec:
        return t * t * self.va + t * self.vd + self.ca

    def flow(self, t: float) -> npvec:
        return 2 * self.va * t + self.vd

    def curve(self, tspace: NDArray) -> NDArray:
        return garr(self.point(t) for t in tspace)

In [ ]:
from math import sqrt, cbrt, cos, acos, pi


def solve(bezier: Bezielt, p: npvec) -> tuple[float, ...]:
    va = bezier.va[:2]
    vd = bezier.vd[:2]
    pc = bezier.ca[:2] - p[:2]

    aa = float(va @ va)
    ad = float(va @ vd)
    dd = float(vd @ vd)
    pa = float(pc @ va)
    pd = float(pc @ vd)

    a_1 = dd + 2 * pa
    p3 = (2 * aa * a_1 - 3 * ad**2) / (12 * aa**2)
    q2 = (ad**3 - aa * ad * a_1 + 2 * aa**2 * pd) / (8 * aa**3)
    off = ad / (2 * aa)

    D = p3**3 + q2**2

    if D > 0:
        C = cbrt(sqrt(D) - q2)
        t_ = C - p3 / C
        return (t_ - off,)
    else:
        rt = sqrt(-p3)
        k = 2 * rt
        ph = 2 * pi / 3
        th = acos(q2 / (p3 * rt)) / 3
        t_0 = k * cos(th)
        t_1 = k * cos(th - ph)
        t_2 = k * cos(th - 2 * ph)
        return (t_0 - off, t_1 - off, t_2 - off)

In [ ]:
def side(bezier: Bezielt, t: float, p: npvec) -> int:
    """+1 for left hand side, -1 for right-hand side"""
    ort = p - bezier.point(t)
    tng = bezier.flow(t)
    crz = ort[1] * tng[0] - ort[0] * tng[1]
    return int(np.sign(crz))

In [ ]:
def horizon(bezier: Bezielt, p: npvec) -> int:
    """number of edges in potential reachability, 0 = OOB"""
    h1 = bezier.v1 @ bezier.v1
    h3 = bezier.v3 @ bezier.v3
    p1 = bezier.v1 @ p
    p3 = bezier.v1 @ p
    return int(p1 < h1) + int(p3 < h3)

In [ ]:
class Projection(NamedTuple):
    smp: npvec
    t: float
    pnt: npvec
    dsq: float = 0
    sgn: int = 0

    @property
    def sdist(self):
        return self.sgn * np.sqrt(self.dsq)

In [ ]:
def project(bezier: Bezielt, sample: npvec) -> Projection:
    ps = bezier.loc(sample)
    tt = solve(bezier, ps)
    if len(tt) == 1:
        t = tt[0]
        pnt = bezier.point(t)
        dsq = disquance(pnt, ps)
    else:
        pnts = tuple(bezier.point(t) for t in tt)
        dsqs = tuple(disquance(p, ps) for p in pnts)
        best = np.argmin(dsqs)
        t = tt[best]
        pnt = pnts[best]
        dsq = dsqs[best]

    sgn = side(bezier, t, ps)

    if -0.5 > t or t > +0.5:
        return Projection(sample, t, bezier.glb(pnt), np.inf, sgn)

    return Projection(sample, t, bezier.glb(pnt), dsq, sgn)
